# Notebook 01 — Core LLM Workflow

Demonstrates provider swap, non-determinism, cache observability, prompting basics, and clean failure surfacing against a real Anthropic backend.

<!-- TODO main-session: expand teaching framing -->


## Setup

Imports and environment loading.


In [ ]:
from __future__ import annotations
import os
import sys
from pathlib import Path

# Add llmops-session to path so `from src.llm import ...` works from notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    load_dotenv = None

if load_dotenv is not None:
    load_dotenv(repo_root / ".env")

from src.llm import LLMClient, LLMCache, CompletionResult

# Confirm we have an Anthropic key (warn but don't fail if mock)
provider = os.getenv("LLM_PROVIDER", "anthropic")
has_key = bool(os.getenv("ANTHROPIC_API_KEY"))
print(f"Provider: {provider} · Anthropic key present: {has_key}")


## Provider abstraction

Same `.complete(...)` call shape, different provider implementations.


In [ ]:
prompt = "In one sentence, what is LLM Ops?"

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — set ANTHROPIC_API_KEY to run]")
else:
    client = LLMClient()
    result = client.complete(prompt, max_tokens=80)
    print("provider:", result.provider)
    print("model:", result.model)
    print("text:", result.text)


In [ ]:
prompt = "In one sentence, what is LLM Ops?"

client = LLMClient(provider="mock")
result = client.complete(prompt, max_tokens=80)
print("provider:", result.provider)
print("model:", result.model)
print("text:", result.text)


In [ ]:
prompt = "In one sentence, what is LLM Ops?"

for provider_name, env_var in [("openai", "OPENAI_API_KEY"), ("gemini", "GOOGLE_API_KEY"), ("ollama", None)]:
    if env_var is not None and not os.getenv(env_var):
        print(f"{provider_name}: [skipped — missing {env_var}]")
        continue

    try:
        client = LLMClient(provider=provider_name)
        result = client.complete(prompt, max_tokens=80)
        preview = " ".join(result.text.split())[:80]
        print(f"{provider_name}: {preview}")
    except ImportError as exc:
        print(f"{provider_name}: [skipped — {exc}]")
    except Exception as exc:
        print(f"{provider_name}: [skipped — {type(exc).__name__}: {exc}]")


The only line that changed across cells 4–6 is the provider name.


## Non-determinism and system messages

Repeated hosted calls can vary, and system messages steer response style.


In [ ]:
variance_prompt = "List three risks of deploying an LLM in production. Be specific."

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no key]")
else:
    client = LLMClient(cache=False)
    for run_number in range(1, 4):
        result = client.complete(variance_prompt, cache=False, temperature=0.9, max_tokens=160)
        print(f"Run {run_number}:")
        print(result.text)
        print()


In [ ]:
user_prompt = "Explain the operational risk of letting an LLM call production tools."
sre_system = "You are a terse senior SRE. Reply in one sentence and use plain language."

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no key]")
else:
    client = LLMClient(cache=False)
    plain = client.complete(user_prompt, cache=False, max_tokens=160)
    steered = client.complete(user_prompt, system=sre_system, cache=False, max_tokens=80)

    print("No system message:")
    print(plain.text)
    print()
    print("With system message:")
    print(steered.text)


## Cache hit / miss / version

Cache events expose misses, hits, and prompt-version invalidation.


In [ ]:
if not os.getenv("ANTHROPIC_API_KEY"):
    cache_demo_ready = False
    print("[skipped — no key]")
else:
    cache_demo_ready = True
    events: list[str] = []
    cache = LLMCache(on_cache_event=lambda status: events.append(status))
    client = LLMClient(cache=cache, prompt_version="v1")

    prompt = "What's the capital of France?"
    r1 = client.complete(prompt, max_tokens=32)  # expect miss
    r2 = client.complete(prompt, max_tokens=32)  # expect hit
    print("After two identical calls:", events)  # ['miss', 'hit']
    print("r1.cache_status:", r1.cache_status, "· r2.cache_status:", r2.cache_status)


In [ ]:
if not globals().get("cache_demo_ready", False):
    print("[skipped — no key]")
else:
    client_v2 = LLMClient(cache=cache, prompt_version="v2")
    r3 = client_v2.complete(prompt, max_tokens=32)  # expect miss (key changed)
    print("After bumping prompt_version:", events)  # ['miss', 'hit', 'miss']
    print("r3.cache_status:", r3.cache_status)
